Error: No connection selected.

In [2]:
# Task 1: Supervised Classification Model

## Customer Churn Prediction using Machine Learning

### Algorithms
1. Logistic Regression
2. Random Forest Classifier

### Evaluation Metrics
- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC

### Additional Evaluation
- 5-Fold Cross Validation
- Confusion Matrix
- ROC Curve

Error: No connection selected.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

Error: No connection selected.

In [5]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Error: No connection selected.

In [6]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Error: No connection selected.

In [ ]:
# ============================================================
# TASK 1 - CUSTOMER CHURN CLASSIFICATION
# Complete Runnable Code
# ============================================================

# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
import warnings

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")


# ============================================================
# 2. LOAD DATASET
# ============================================================

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(url)

print("=" * 70)
print("DATASET LOADED")
print("=" * 70)

print("Dataset Shape:", df.shape)
print("\nFirst 5 Rows:")
display(df.head())


# ============================================================
# 3. BASIC DATA INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())


# ============================================================
# 4. DATA PREPROCESSING
# ============================================================

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Remove missing values
df = df.dropna()

# Remove customer ID
df = df.drop("customerID", axis=1)

# Encode target variable
df["Churn"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

print("\n" + "=" * 70)
print("DATA PREPROCESSING COMPLETED")
print("=" * 70)

print("Final Dataset Shape:", df.shape)
print("\nTarget Distribution:")
print(df["Churn"].value_counts())


# ============================================================
# 5. EDA - CHURN DISTRIBUTION
# ============================================================

plt.figure(figsize=(7, 5))

sns.countplot(
    x="Churn",
    data=df
)

plt.title("Customer Churn Distribution")
plt.xlabel("Churn (0 = No, 1 = Yes)")
plt.ylabel("Number of Customers")
plt.show()


# ============================================================
# 6. CHURN PERCENTAGE
# ============================================================

churn_percentage = df["Churn"].value_counts(
    normalize=True
) * 100

print("\nChurn Percentage:")
print(churn_percentage.round(2))


# ============================================================
# 7. CONTRACT VS CHURN
# ============================================================

plt.figure(figsize=(8, 5))

sns.countplot(
    x="Contract",
    hue="Churn",
    data=df
)

plt.title("Customer Churn by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Number of Customers")
plt.xticks(rotation=20)

plt.show()


# ============================================================
# 8. MONTHLY CHARGES VS CHURN
# ============================================================

plt.figure(figsize=(8, 5))

sns.boxplot(
    x="Churn",
    y="MonthlyCharges",
    data=df
)

plt.title("Monthly Charges vs Churn")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")

plt.show()


# ============================================================
# 9. TENURE DISTRIBUTION
# ============================================================

plt.figure(figsize=(8, 5))

sns.histplot(
    data=df,
    x="tenure",
    hue="Churn",
    bins=30,
    kde=True
)

plt.title("Tenure Distribution by Churn")
plt.xlabel("Tenure")
plt.ylabel("Number of Customers")

plt.show()


# ============================================================
# 10. PREPARE FEATURES AND TARGET
# ============================================================

X = df.drop("Churn", axis=1)
y = df["Churn"]

print("\nFeatures Shape:", X.shape)
print("Target Shape:", y.shape)


# ============================================================
# 11. TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\n" + "=" * 70)
print("TRAIN TEST SPLIT")
print("=" * 70)

print("Training Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])


# ============================================================
# 12. IDENTIFY FEATURE TYPES
# ============================================================

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("\nNumerical Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)


# ============================================================
# 13. PREPROCESSING PIPELINE
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)


# ============================================================
# 14. LOGISTIC REGRESSION
# ============================================================

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

logistic_model.fit(
    X_train,
    y_train
)

y_pred_lr = logistic_model.predict(X_test)

y_prob_lr = logistic_model.predict_proba(
    X_test
)[:, 1]

print("\nLogistic Regression trained successfully!")


# ============================================================
# 15. RANDOM FOREST
# ============================================================

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                class_weight="balanced"
            )
        )
    ]
)

random_forest_model.fit(
    X_train,
    y_train
)

y_pred_rf = random_forest_model.predict(X_test)

y_prob_rf = random_forest_model.predict_proba(
    X_test
)[:, 1]

print("Random Forest trained successfully!")


# ============================================================
# 16. EVALUATION FUNCTION
# ============================================================

def evaluate_model(y_true, y_pred, y_probability):

    return {
        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "Precision": precision_score(
            y_true,
            y_pred
        ),

        "Recall": recall_score(
            y_true,
            y_pred
        ),

        "F1 Score": f1_score(
            y_true,
            y_pred
        ),

        "ROC-AUC": roc_auc_score(
            y_true,
            y_probability
        )
    }


# ============================================================
# 17. MODEL EVALUATION
# ============================================================

logistic_results = evaluate_model(
    y_test,
    y_pred_lr,
    y_prob_lr
)

random_forest_results = evaluate_model(
    y_test,
    y_pred_rf,
    y_prob_rf
)

results = pd.DataFrame(
    [
        logistic_results,
        random_forest_results
    ],
    index=[
        "Logistic Regression",
        "Random Forest"
    ]
)

print("\n" + "=" * 70)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 70)

display(results.round(4))


# ============================================================
# 18. MODEL COMPARISON GRAPH
# ============================================================

results.plot(
    kind="bar",
    figsize=(11, 6)
)

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xlabel("Model")
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


# ============================================================
# 19. LOGISTIC REGRESSION CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 70)
print("LOGISTIC REGRESSION CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_test,
        y_pred_lr,
        target_names=[
            "No Churn",
            "Churn"
        ]
    )
)


# ============================================================
# 20. RANDOM FOREST CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 70)
print("RANDOM FOREST CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_test,
        y_pred_rf,
        target_names=[
            "No Churn",
            "Churn"
        ]
    )
)


# ============================================================
# 21. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred_rf
)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[
        "No Churn",
        "Churn"
    ],
    yticklabels=[
        "No Churn",
        "Churn"
    ]
)

plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()


# ============================================================
# 22. ROC CURVE
# ============================================================

fpr_lr, tpr_lr, _ = roc_curve(
    y_test,
    y_prob_lr
)

fpr_rf, tpr_rf, _ = roc_curve(
    y_test,
    y_prob_rf
)

lr_auc = roc_auc_score(
    y_test,
    y_prob_lr
)

rf_auc = roc_auc_score(
    y_test,
    y_prob_rf
)

plt.figure(figsize=(8, 6))

plt.plot(
    fpr_lr,
    tpr_lr,
    label=f"Logistic Regression (AUC = {lr_auc:.3f})"
)

plt.plot(
    fpr_rf,
    tpr_rf,
    label=f"Random Forest (AUC = {rf_auc:.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")

plt.legend()
plt.show()


# ============================================================
# 23. 5-FOLD CROSS VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

lr_cv_scores = cross_val_score(
    logistic_model,
    X,
    y,
    cv=cv,
    scoring="roc_auc"
)

rf_cv_scores = cross_val_score(
    random_forest_model,
    X,
    y,
    cv=cv,
    scoring="roc_auc"
)

print("\n" + "=" * 70)
print("5-FOLD CROSS VALIDATION")
print("=" * 70)

print(
    "\nLogistic Regression CV Scores:",
    np.round(lr_cv_scores, 4)
)

print(
    "Logistic Regression Mean:",
    round(lr_cv_scores.mean(), 4)
)

print(
    "\nRandom Forest CV Scores:",
    np.round(rf_cv_scores, 4)
)

print(
    "Random Forest Mean:",
    round(rf_cv_scores.mean(), 4)
)


# ============================================================
# 24. CROSS VALIDATION COMPARISON
# ============================================================

cv_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],

    "Mean CV ROC-AUC": [
        lr_cv_scores.mean(),
        rf_cv_scores.mean()
    ],

    "Std CV ROC-AUC": [
        lr_cv_scores.std(),
        rf_cv_scores.std()
    ]
})

print("\nCross-Validation Comparison:")
display(cv_results.round(4))


# ============================================================
# 25. CROSS VALIDATION GRAPH
# ============================================================

plt.figure(figsize=(8, 5))

plt.bar(
    cv_results["Model"],
    cv_results["Mean CV ROC-AUC"]
)

plt.ylim(0, 1)

plt.ylabel("Mean ROC-AUC")
plt.xlabel("Model")

plt.title("5-Fold Cross-Validation Performance")

plt.show()


# ============================================================
# 26. SELECT BEST MODEL
# ============================================================

best_model_name = results["ROC-AUC"].idxmax()

if best_model_name == "Logistic Regression":
    best_model = logistic_model
else:
    best_model = random_forest_model

print("\n" + "=" * 70)
print("FINAL MODEL SELECTION")
print("=" * 70)

print("Best Model:", best_model_name)


# ============================================================
# 27. FINAL METRICS
# ============================================================

best_metrics = results.loc[
    best_model_name
]

print("\nFinal Test Metrics:")

for metric, value in best_metrics.items():
    print(
        f"{metric}: {value:.4f}"
    )


# ============================================================
# 28. SAVE MODEL
# ============================================================

os.makedirs(
    "model",
    exist_ok=True
)

model_path = "model/customer_churn_model.pkl"

joblib.dump(
    best_model,
    model_path
)

print("\nModel saved successfully!")
print("Location:", model_path)


# ============================================================
# 29. FINAL RESULT
# ============================================================

print("\n" + "=" * 70)
print("TASK 1 COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\nDataset Shape:", df.shape)

print("Best Model:", best_model_name)

print("\nAccuracy:", round(best_metrics["Accuracy"], 4))
print("Precision:", round(best_metrics["Precision"], 4))
print("Recall:", round(best_metrics["Recall"], 4))
print("F1 Score:", round(best_metrics["F1 Score"], 4))
print("ROC-AUC:", round(best_metrics["ROC-AUC"], 4))

print("\nModel File:")
print("model/customer_churn_model.pkl")

print("\nReady for Task 2 - API and Docker!")